# Схожесть текстов через `text_similarity_datasketch.py`

Этот ноутбук использует готовый модуль `text_similarity_datasketch.py`: импортирует из него `compare_texts`, `preprocess` и `similarity_label`, а затем визуализирует результат. Логика MinHash находится в `.py` файле, ноутбук отвечает только за демонстрацию.


## Зависимости

Если импорт ниже падает на `datasketch`, установите пакеты:

```bash
python3 -m pip install datasketch matplotlib matplotlib-venn
```

`matplotlib-venn` опционален: без него будет показана столбчатая диаграмма.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

try:
    from matplotlib_venn import venn2
except ImportError:
    venn2 = None

# Импорт работает и при запуске ноутбука из lections/text_similarity,
# и при запуске Jupyter из корня репозитория.
module_dir = Path.cwd()
if not (module_dir / "text_similarity_datasketch.py").exists():
    module_dir = Path.cwd() / "lections" / "text_similarity"

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from text_similarity_datasketch import compare_texts, preprocess, similarity_label


## Тексты и параметры

Можно менять строки, размер n-грамм и длину MinHash-сигнатуры.


In [ ]:
text1 = "Машинное обучение анализирует данные и строит модели"
text2 = "Машинное обучение изучает данные и улучшает модели"

ngram_size = 1
num_perm = 128


## Сравнение текстов функцией из модуля

`compare_texts` возвращает множества n-грамм, точный коэффициент Жаккара и приближенную MinHash-оценку.


In [ ]:
result = compare_texts(text1, text2, n=ngram_size, num_perm=num_perm)

print("Токены 1:", preprocess(text1))
print("Токены 2:", preprocess(text2))
print("n-граммы 1:", sorted(result.text1_ngrams))
print("n-граммы 2:", sorted(result.text2_ngrams))
print(f"Точный Жаккар: {result.exact_jaccard:.3f}")
print(f"MinHash-оценка Жаккара: {result.minhash_jaccard:.3f}")
print("Вывод:", similarity_label(result.minhash_jaccard))


## Пересечение n-грамм


In [ ]:
ngrams1 = result.text1_ngrams
ngrams2 = result.text2_ngrams

intersection = ngrams1 & ngrams2
only_text1 = ngrams1 - ngrams2
only_text2 = ngrams2 - ngrams1

print("Только текст 1:", sorted(only_text1))
print("Пересечение:", sorted(intersection))
print("Только текст 2:", sorted(only_text2))


## Визуализация множеств n-грамм


In [ ]:
plt.figure(figsize=(7, 5))

if venn2 is not None:
    venn2([ngrams1, ngrams2], set_labels=("Текст 1", "Текст 2"))
    plt.title("Пересечение множеств n-грамм")
else:
    labels = ["Только текст 1", "Пересечение", "Только текст 2"]
    values = [len(only_text1), len(intersection), len(only_text2)]
    plt.bar(labels, values, color=["#4C78A8", "#59A14F", "#F28E2B"])
    plt.ylabel("Количество n-грамм")
    plt.title("Состав множеств n-грамм")

plt.show()


## Точный Жаккар vs MinHash

На графике видно, насколько оценка MinHash близка к точному коэффициенту Жаккара для выбранного `num_perm`.


In [ ]:
scores = [result.exact_jaccard, result.minhash_jaccard]
labels = ["Точный Жаккар", "MinHash"]
colors = ["#D62728", "#4C78A8"]

plt.figure(figsize=(6, 4))
plt.bar(labels, scores, color=colors)
plt.ylim(0, 1)
plt.ylabel("Сходство")
plt.title("Сравнение точной и MinHash-оценки")

for idx, score in enumerate(scores):
    plt.text(idx, min(score + 0.02, 0.98), f"{score:.3f}", ha="center")

plt.show()


## Как `num_perm` влияет на оценку

Здесь ноутбук снова вызывает `compare_texts` из `text_similarity_datasketch.py` для разных длин сигнатуры.


In [ ]:
perm_values = [16, 32, 64, 128, 256, 512]
estimates = [
    compare_texts(text1, text2, n=ngram_size, num_perm=perm).minhash_jaccard
    for perm in perm_values
]

plt.figure(figsize=(8, 4.5))
plt.plot(perm_values, estimates, marker="o", label="MinHash")
plt.axhline(result.exact_jaccard, color="#D62728", linestyle="--", label="Точный Жаккар")
plt.xscale("log", base=2)
plt.xticks(perm_values, perm_values)
plt.ylim(0, 1)
plt.xlabel("num_perm")
plt.ylabel("Сходство")
plt.title("Приближение MinHash к коэффициенту Жаккара")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

for perm, score in zip(perm_values, estimates):
    print(f"num_perm={perm:>3}: {score:.3f}")


## Итог


In [ ]:
print(f"Сходство по MinHash: {result.minhash_jaccard:.3f}")
print("Вывод:", similarity_label(result.minhash_jaccard))
